## **HECHOS_STOCK**

In [29]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
hechos_stock = spark.read.table("lh_retailnova_silver_dev.hechos_stock")
display(hechos_stock)


StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c987b398-0776-4406-bdd4-0117a2678f75)

## **HECHOS_TRANSACIONES**

In [30]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
hechos_venta = spark.read.table("lh_retailnova_silver_dev.hechos_transaccion")
display(hechos_venta)


StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0534bcb-c638-4fae-9c3b-beb83c581adc)

In [31]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window


fecha_actual = F.current_date()
fecha_180_dias = F.date_sub(fecha_actual, 180)


rfm_base = hechos_venta.filter(F.col("fecha_trans") >= fecha_180_dias) \
    .groupBy("id_miembro").agg(
        F.datediff(fecha_actual, F.max("fecha_trans")).alias("recency_dias"),
        F.count("fecha_trans").alias("frequency_tx"), 
        F.sum("total_venta").alias("monetary_valor")  
    )


window_r = Window.orderBy(F.col("recency_dias").desc(), F.col("id_miembro"))
window_f = Window.orderBy(F.col("frequency_tx").asc(), F.col("id_miembro"))
window_m = Window.orderBy(F.col("monetary_valor").asc(), F.col("id_miembro"))


fact_rfm_clientes = rfm_base.withColumn(
    "score_r", F.ntile(5).over(window_r)
).withColumn(
    "score_f", F.ntile(5).over(window_f)
).withColumn(
    "score_m", F.ntile(5).over(window_m)
).withColumn(
    "segmento_rfm_cod",
    F.concat(F.lit("R"), F.col("score_r"), 
             F.lit("-F"), F.col("score_f"), 
             F.lit("-M"), F.col("score_m"))
).withColumn(
    "segmento_nombre",
   
    F.when(F.col("segmento_rfm_cod") == "R5-F5-M5", "Champions") 
     .when(F.col("score_r") == 1, "En Riesgo / Perdidos")
     .otherwise("Regular")
)


display(fact_rfm_clientes.orderBy(F.col("segmento_rfm_cod").desc()))

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2a3ede2a-2d33-457a-b5b8-ad46e302dab1)

In [32]:
distribucion_clientes = fact_rfm_clientes.groupBy("segmento_nombre").count()

display(distribucion_clientes)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0e5a889-17e2-4109-8e4e-3eeaf1348f4b)

In [33]:
display(fact_rfm_clientes)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6af85dfe-6f2a-43ca-962c-90317fedd065)

## **DIM_ARTICULO**

In [48]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_articulo = spark.read.table("lh_retailnova_silver_dev.articulo")
display(dim_articulo)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d64d2e0f-4a1e-4fc5-8169-c3ad59cad4cd)

## **DIM_TIENDA**

In [34]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_tienda= spark.read.table("lh_retailnova_silver_dev.dim_tienda")
display(dim_tienda)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 23513c05-ee4c-45f7-99c3-8ac12a78216d)

## **DIM_PROVEEDOR**

In [35]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_proveedor = spark.read.table("lh_retailnova_silver_dev.dim_proveedor")
display(dim_proveedor)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4243967-5340-4a9c-b737-265acd291cf5)

## **HECHOS_DEVOLUCION**


In [36]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
hechos_devolucion= spark.read.table("lh_retailnova_silver_dev.hechos_devolucion")
display(hechos_devolucion)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54231196-312b-4072-baf9-4036bbcee073)

## **DIM_PAIS**

In [37]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_pais = spark.read.table("lh_retailnova_silver_dev.dim_pais")
display(dim_pais)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b936b61c-10d1-4377-9cd1-17ec9e14b62e)

## **DIM_CIUDAD**

In [38]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_ciudad = spark.read.table("lh_retailnova_silver_dev.dim_ciudad")
display(dim_ciudad)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a7b2b5c9-ae0e-4b58-866a-7a3a985b91a3)

## **DIM_MIEMBROS**

In [39]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_miembros = spark.read.table("lh_retailnova_silver_dev.dim_miembros")
display(dim_miembros)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 380d3292-8a1e-40cc-acba-24781c68ae3f)

In [40]:
import pyspark.sql.functions as F

dim_miembros = dim_miembros.withColumn(
    "segmento_edad",
    
    F.when(F.col("edad").isin("18-20", "21-25", "26-30", "31-35"), "Joven")
    
    
     .when(F.col("edad").isin("36-40", "41-50", "51-60"), "Adulto")
     
    
     .when(F.col("edad") == "61+", "Adulto Mayor")
     
    
     .otherwise("No Informado") 
)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 41, Finished, Available, Finished, False)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, -1, Finished, Available, Finished, True)

## **DIM_SUB_CATEGORIA**

In [42]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_sub_categoria = spark.read.table("lh_retailnova_silver_dev.dim_sub_categoria")
display(dim_sub_categoria)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3adf5c2a-1442-434c-8847-82e8514778a9)

## **DIM_CATEGORIA**

In [43]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_categoria  = spark.read.table("lh_retailnova_silver_dev.dim_categoria")
display(dim_categoria)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4069d483-29e4-4f5d-8fa0-6ca105350446)

## **DIM_MACRO_CATEGORIA**

In [44]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
dim_macro_categoria= spark.read.table("lh_retailnova_silver_dev.dim_macro_categoria")
display(dim_macro_categoria)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cbd25d32-79af-4882-a252-65ee5fc0c3ac)

## **kPI**

In [45]:
 display(dim_miembros)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4c1cc54-7aa7-409a-9e6f-6e4d9e670603)

**VENTAS POR SEGMENTO DE EDAD**

In [46]:
import pyspark.sql.functions as F

# 1. Cruzar las ventas con los miembros para traer el "segmento_edad"
ventas_con_edad = hechos_venta.join(
    dim_miembros.select("id_miembro", "segmento_edad"), 
    on="id_miembro", 
    how="inner" 
)

# 2. Calcular los KPIs agrupando por el segmento de edad
kpi_compras_por_edad = ventas_con_edad.groupBy("segmento_edad").agg(
    
    F.count("id_miembro").alias("total_transacciones"), 
    F.sum("total_venta").alias("ingresos_totales"), 
    F.avg("total_venta").alias("ticket_promedio")
)

# 3. Ordenar los resultados de mayor a menor
kpis_ordenados = kpi_compras_por_edad.orderBy(F.col("ingresos_totales").desc())

# Mostrar el resultado final
display(kpis_ordenados)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 01c73bf1-ae1a-40a0-8ac2-4066670fbd08)

**VENTAS POR MACRO CATEGORIA**

In [57]:
import pyspark.sql.functions as F


ventas_articulos = hechos_venta.join(
    dim_articulo.select("articulo_id", "id_categ_n1"),
    hechos_venta["id_articulo"] == dim_articulo["articulo_id"],
    how="inner"
)


ventas_con_nombres = ventas_articulos.join(
    dim_macro_categoria.select("id_categ_n1", "macro_categoria"),
    on="id_categ_n1",
    how="inner"
)


kpi_macrocategoria = ventas_con_nombres.groupBy("id_categ_n1", "macro_categoria").agg(
    F.sum("cantidad_vendida").alias("total_unidades_vendidas"),
    F.round(F.sum("total_venta"), 2).alias("ingresos_totales")
)

kpi_macrocategoria = kpi_macrocategoria.orderBy(F.col("ingresos_totales").desc())


display(kpi_macrocategoria)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72a5ce0f-7b9a-43ef-93bb-8a844ce2c02d)

**VENTAS POR GENERO**

In [61]:
import pyspark.sql.functions as F


ventas_genero =hechos_venta.join(
    dim_miembros.select("id_miembro", "genero"),
    on="id_miembro",
    how="inner"
)


kpi_genero = ventas_genero.groupBy("genero").agg(
    
 
    F.countDistinct("id_transaccion").alias("total_transacciones"),
    

    F.sum("cantidad_vendida").alias("total_unidades_compradas"),
    
  
    F.round(F.sum("total_venta"), 2).alias("ingresos_totales")
)


kpi_genero = kpi_genero.orderBy(F.col("ingresos_totales").desc())


display(kpi_genero)

StatementMeta(, 67e4ec1c-c126-4b7d-822c-c4862ddcc4ef, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff0d17d8-1713-4efa-acfb-f7cd6079dd2d)